# 学习率热身（Learning Rate Warmup）
- 是现代深度学习模型训练中一种非常关键且广泛使用的优化策略。
- 它的核心思想是：
  - <font color='red'>在训练的初始阶段，使用一个较小的学习率，并在预设的步数内逐渐将其增加到预设的目标学习率。</font>

- 为了让你更直观地理解，我们可以用生活中的例子来类比： 
  - 就像开车起步时，如果直接猛踩油门，车辆可能会失控；
  - 或者像**健身前需要慢跑热身**，直接全力冲刺容易导致肌肉拉伤。
  - Warmup 就是给神经网络一个“温柔启动”的缓冲期。

以下是对学习率预热的详细解析：

## 为什么需要学习率预热？（底层机制）
<font color='red'>在训练初期，模型面临几个特殊问题</font>，**直接使用较大的目标学习率会导致训练不稳定（如损失剧烈波动甚至发散）**。预热的底层机制主要体现在以下几个方面：

### 稳定自适应优化器的内部统计：
  - 像 Adam、RMSProp 这样的自适应优化器，**内部需要维护梯度矩的估计值**。
    - <font color='red'>在训练刚开始时，这些估计值仅基于极少的样本，存在很大噪声。</font>预热期允许优化器在较小的学习率下，先收集足够的数据以计算出合理的方差和梯度统计量，避免第一步更新就把网络推向错误的方向。
  
### 缓解随机初始化的梯度冲击：
  - 网络权重刚初始化时，前几轮反向传播计算出的初始梯度可能极大或指向不稳定的方向。预热通过小步长让模型参数温和调整，迫使网络进入损失景观（Loss Landscape）中更良性的区域，从而让网络后续能够承受更大的目标学习率。
  
### 适应大批量训练：
  - 在使用非常大的小批量（Large Batch）进行训练时，损失函数往往更陡峭，预热有助于减轻大学习率带来的不稳定性。
  
### 复杂架构的必需品：
  - 对于 Transformer 等复杂架构，预热几乎已成为标准做法，它能显著提升初始训练阶段的可靠性。
  
## 常见的预热策略
预热阶段的持续时间（warmup_steps）是一个超参数。在预热阶段结束后，学习率达到目标值，通常会无缝衔接标准的学习率衰减策略（如阶梯衰减、余弦退火等）。常见的预热曲线包括：
- 线性预热 (Linear Warmup)：
  - 最基础、最常用的方式。学习率从 0（或极小值）随步数线性增长到目标值。BERT、GPT 等模型常用此策略。
- 余弦预热 (Cosine Warmup)：
  - 学习率按照余弦曲线平滑增长，过渡更加平缓，常用于视觉任务（如 CNN、ResNet）。
- 恒定预热 (Constant Warmup)：
  - 在预热步数内使用一个较小的恒定学习率，预热结束后突然跳跃到目标学习率。这种变体相对少见。
- 指数/平方根预热：
  - 初期增长较快，后期趋缓，适合配合特定的自适应优化器使用。
  
## 学习率预热的核心优势
- 提升训练稳定性：
  - 有效抑制初始阶段的 Loss Spike（损失尖峰），实测显示启用 Warmup 后首个 epoch 的损失波动幅度可大幅降低。
- 加速收敛与提升性能：
  - 预热后模型能以更大的学习率快速收敛。实验表明，使用 Warmup 技巧可以将模型的收敛速度提高约 20%，并将最终准确率提高约 1%。
- 防止早期过拟合：
  - 初始阶段的小学习率有助于模型在数据分布上进行平稳“探索”，避免过早过拟合到局部特征。
- 增强超参数调优的鲁棒性：
  - 预热带来的稳定性使得模型能够安全地使用更大的目标学习率，这反过来使得超参数调优变得更加稳健。
  
## 实际工程应用
在现代深度学习框架（如 PyTorch）中，实现 Warmup 非常简单。通常通过组合调度器（如 LambdaLR 或 SequentialLR）来实现：先定义一个线性预热调度器，再链接一个衰减调度器（如 StepLR 或 CosineAnnealingLR）。

### 总结：
- 学习率预热就像是给深度学习模型穿上了一层“减震器”。它通过牺牲训练最初期的微小步长，换取了整个训练过程的极度稳定和最终性能的显著提升。在当前的工业界（如 YOLO 目标检测、大语言模型训练），“Warmup + Cosine退火”已经成为事实上的标准训练范式。

# Kimi

## 什么是学习率热身？
定义：<font color='red'>在训练开始的若干步（warmup steps）内，学习率从一个很小的值（通常是 0）逐渐上升到设定的峰值，而不是一开始就使用全量学习率。</font>

In [ ]:
无热身:  lr = 1e-4 ─────────────────────────────→
有热身:  lr = 0 ──→ 2e-5 ──→ 5e-5 ──→ 8e-5 ──→ 1e-4 ──→ ...
         ↑______ Warmup Phase ______↑

## 为什么需要热身？
###  训练初期的梯度不稳定
<font color='red'>模型参数在初始化时是随机的，**前向传播的输出分布与最终目标差异极大**</font>此时：
- 梯度估计方差极大（基于小 batch 的统计噪声）
- 如果直接用大学习率，**参数会在高维空间中剧烈震荡**
- **可能导致损失爆炸或陷入糟糕的局部最优**


###  Adam/AdamW 优化器的二阶矩偏差
Adam 类优化器维护一阶矩（动量）和二阶矩（梯度平方的指数移动平均）：
- $m_t=β_1m_{t−1}+(1−β_1)g_t$,
- $v_t=β_2v_{t−1}+(1−β_2)g^2_t$

 
在训练初期：
- **m_t和 v_t都偏向 0**（因为初始化为 0，且 **β  接近 1**）
- 更新量$ Δθ= \frac{m_t}{\sqrt{v_t+ϵ}}$ 在初期被系统性低估
- <font color='red'>热身通过放大学习率来补偿这个偏差</font>

###  Transformer/大模型的特殊敏感性
大模型（如 LLM）的深层结构对初始更新特别敏感：
- 注意力机制中的 softmax **在初期可能进入极端饱和区**
- LayerNorm 的统计量尚未稳定
- 一个过大的初始步可能永久破坏某些层的表示能力

## 常见的热身策略

| 策略          | 公式                                                                   | 特点          |
| :---------- | :------------------------------------------------------------------- | :---------- |
| **线性热身**    | $\text{lr} = \text{base\_lr} \times \frac{t}{T_{\text{warmup}}}$     | 最常用，简单稳定    |
| **指数热身**    | $\text{lr} = \text{base\_lr} \times (\frac{t}{T_{\text{warmup}}})^2$ | 初期增长更慢，后期加速 |
| **常数热身**    | 前 $T$ 步固定小 lr，之后跳变                                                   | 极少使用        |
| **热身 + 衰减** | 热身后接 cosine/linear decay                                             | <font color='red'>现代 LLM 训练标配 </font>|

## PyTorch 中的实现方式
### 方式 1：手动实现（最直观）

In [ ]:
import torch
from torch.optim import AdamW

model = ...
optimizer = AdamW(model.parameters(), lr=1e-4)

base_lr = 1e-4
warmup_steps = 1000
total_steps = 100000

for step in range(total_steps):
    # 1. 计算当前学习率
    if step < warmup_steps:
        lr = base_lr * (step / warmup_steps)  # 线性热身
    else:
        # 热身后可接衰减（如余弦）
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        lr = base_lr * 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159)))
    
    # 2. 更新优化器中的学习率
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    # 3. 正常训练
    loss = ...
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

### 方式 2：LambdaLR（推荐）

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    # 热身后保持恒定（或接其他衰减）
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda)

for step in range(total_steps):
    ...
    optimizer.step()
    scheduler.step()  # 每次更新后调用

### 方式 3：Transformers 库（Hugging Face）

In [ ]:
from transformers import get_cosine_schedule_with_warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=1000,      # 热身步数
    num_training_steps=100000,  # 总训练步数
    num_cycles=0.5,             # 余弦周期数（0.5 = 半周期）
)

# 训练循环中
for step in range(total_steps):
    ...
    optimizer.step()
    scheduler.step()

#### 相关函数家族：

In [ ]:
from transformers import (
    get_linear_schedule_with_warmup,      # 线性衰减
    get_cosine_schedule_with_warmup,      # 余弦衰减
    get_cosine_with_hard_restarts_schedule_with_warmup,  # 余弦+重启
    get_polynomial_decay_schedule_with_warmup,           # 多项式衰减
    get_constant_schedule_with_warmup,                  # 热身后恒定
)

### 方式 4：OneCycleLR（快速收敛）

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

scheduler = OneCycleLR(
    optimizer,
    max_lr=1e-4,
    total_steps=total_steps,
    pct_start=0.1,        # 前 10% 为热身阶段
    anneal_strategy='cos',
    div_factor=25.0,      # 初始 lr = max_lr / 25
    final_div_factor=1000.0,  # 最终 lr = max_lr / 1000
)

## 完整实战代码（LLM 预训练风格）

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

# 假设
model = MyTransformer()
optimizer = AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.95), weight_decay=0.1)

# 训练配置
batch_size = 32
dataset_size = 1000000
total_steps = dataset_size // batch_size  # 约 31250
warmup_steps = 2000  # 通常占总步数的 1%~10%

# 学习率调度器
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# 训练循环
model.train()
for step, batch in enumerate(dataloader):
    # 前向
    logits = model(batch['input_ids'])
    loss = compute_loss(logits, batch['labels'])
    
    # 反向
    loss.backward()
    
    # 梯度裁剪（LLM 训练标配）
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # 更新
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    
    # 日志
    if step % 100 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Step {step}: loss={loss.item():.4f}, lr={current_lr:.2e}")

## 在 RLHF/RLVR 中的特殊考量
### 热身步数通常更短

| 场景            | 典型 Warmup 比例 | 原因                    |
| :------------ | :----------- | :-------------------- |
| **预训练**       | 1%~5% 总步数    | 从零开始，需要充分预热           |
| **SFT**       | 1%~3%        | 基于预训练权重，已有基础表示        |
| **RLHF/RLVR** | **0.5%~2%**  | <font color='red'>基于 SFT 模型初始化，且数据量通常较小</font> |

### 与 KL 惩罚的协同
在 RL 中，学习率过高可能导致策略偏离参考模型太快，KL 散度爆炸。热身帮助：
- 初期小步探索，稳定重要性采样比率
- 避免初期因学习率过大导致 ratio 剧烈波动

### 与梯度裁剪的配合

In [ ]:
# RL 训练中的典型配置
optimizer = AdamW(policy_model.parameters(), lr=1e-6)  # RL 的 lr 通常比 SFT 小 10~100 倍
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, ...)

for step in range(rl_steps):
    ...
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

## 常见误区

| 误区                            | 正确做法                                                 |
| :---------------------------- | :--------------------------------------------------- |
| "热身步数越多越好"                    | 过长热身会**浪费计算**，且模型可能在低 lr 区"预热过度"，错过快速学习期。通常 1%~5% 足够 |
| "热身后学习率应该恒定"                  | 现代 LLM 训练通常热身后接**衰减**（cosine decay），让模型在后期精细调整       |
| "热身只用于大模型"                    | <font color='red'>任何使用 Adam + Transformer 的场景都建议热身，小模型同样受益 </font>            |
| "scheduler.step() 放在 epoch 后" | 应该放在 **optimizer.step() 之后、每个 step 调用**，不是每个 epoch   |


## 一句话总结

学习率热身是训练初期的"缓踩油门"——<font color='red'>让模型在参数空间中以小步试探，待梯度统计稳定后再全速前进</font>。在 PyTorch 中，通过 LambdaLR 或 transformers 的 schedule 函数即可轻松实现，是现代 LLM 训练不可或缺的基础设施。